# Performance profile

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are ~285,000
NetCDF files. The [englacial/ismip7](https://source.coop/englacial/ismip7) Icechunk
stores hold *virtual references* into those files: no data is copied, every group
(`<ESM>/<experiment>/<product>`) is one xarray Dataset with a normalised time axis,
exact standard-grid coordinates and a correct `crs`. Reading is anonymous.

Set `ISMIP7_STORE_AIS` / `ISMIP7_STORE_GRIS` to `local:/path/to/repo` to run these
notebooks against a local dev store.


In [1]:
import os
import time

from ismip7_helper.access import list_groups, store_target


def read(a, tries=3):
    """Load a lazy array and return the seconds taken; retry on transient object-store errors."""
    for attempt in range(tries):
        t0 = time.time()
        try:
            a.load()
            return round(time.time() - t0, 2)
        except Exception as e:
            if attempt == tries - 1:
                raise
            print("retry after", type(e).__name__)


REGION = os.environ.get("ISMIP7_NB_REGION", "AIS")
print("store:", store_target(REGION).url)
groups = list_groups(REGION)
print(len(groups), "groups; first:", groups[:5])
GROUP = os.environ.get("ISMIP7_NB_GROUP", groups[0])

store: local:/tmp/claude-0/-home-user-ismip7-indexing/c378e836-2097-52d9-9f11-06a02384e3a8/scratchpad/devstore/icechunk-ais


  2026-11-01T00:57:35.073374Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



2 groups; first: ['CESM2-WACCM/historical/SDBN1-8000m', 'CESM2-WACCM/historical/ocean']


What a reader pays: cold open of the repository, opening one group, first chunk, and a full-year read; plus how big the manifests are (what icechunk-js in a browser must download before the first timestep).

In [2]:
import pandas as pd
import xarray as xr

from ismip7_helper.access import open_repo

results = []
for gp in groups[:10]:
    t0 = time.time()
    sess = open_repo(REGION).readonly_session(branch="main")
    t_open = round(time.time() - t0, 2)
    t0 = time.time()
    ds = xr.open_zarr(sess.store, group=gp, consolidated=False)
    t_group = round(time.time() - t0, 2)
    var = next(v for v in ds.data_vars if ds[v].ndim >= 3)
    t_first = read(ds[var].isel(time=0))
    n = min(12, ds.sizes["time"])
    t_year = read(ds[var].isel(time=slice(0, n)))
    results.append(
        {
            "group": gp,
            "var": var,
            "steps": ds.sizes["time"],
            "chunks": ds[var].encoding.get("chunks"),
            "open": t_open,
            "group_open": t_group,
            "first_step": t_first,
            "twelve_steps": t_year,
        }
    )
pd.DataFrame(results)

  2026-11-01T00:57:36.397409Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



  2026-11-01T00:57:37.941415Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



retry after StorageError


retry after StorageError


,group,var,steps,chunks,open,group_open,first_step,twelve_steps
0,CESM2-WACCM/historical/SDBN1-8000m,acabf,36,"(1, 761, 761)",0.0,0.03,0.63,0.89
1,CESM2-WACCM/historical/ocean,tf,20,"(1, 4, 96, 96)",0.0,0.02,7.86,87.41


## Manifest sizes

Icechunk splits each array's manifest every 120 chunks along time; a client reads one split per timestep it touches.

In [3]:
repo = open_repo(REGION)
sess = repo.readonly_session(branch="main")
sizes = sorted((m.size_bytes for m in repo.list_manifest_files(sess.snapshot_id)), reverse=True)
print(
    f"{len(sizes)} manifests; total {sum(sizes) / 1e6:.1f} MB; "
    f"largest {sizes[0] / 1e6:.2f} MB; median {sizes[len(sizes) // 2] / 1e3:.0f} kB"
)

10 manifests; total 0.1 MB; largest 0.14 MB; median 0 kB


  2026-11-01T00:59:47.078612Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

